# GrepSeek — interactive demo (Colab)

[GrepSeek](https://github.com/alirezasalemi7/grepseek) answers questions by **searching a raw Wikipedia corpus with shell commands** (`rg`/`grep`) — *Direct Corpus Interaction* — instead of a dense/sparse index. This notebook spins up the **GRPO** model with vLLM, points the repo's own agent harness at it, and lets you ask questions and watch the agent search.

> **Runtime requirements (read first).**
> - **GPU:** the model is **9B** (bf16 ≈ 18 GB). Best on a Colab **A100 (40 GB)** or **L4 (24 GB)** — *Runtime → Change runtime type → GPU*. On a free **T4 (16 GB)**, set `SERVE_MODE = "4bit"` in Step 3 to load it in 4-bit (slower, demo-quality).
> - **Disk:** the full corpus is **~14 GB** on disk (≈5 GB download). Colab gives ~100 GB, so this fits — but on Colab CPU each `rg` over 14 GB takes ~10–30 s, so a query with a few tool calls takes ~1 min. If that's too slow or disk is tight, use the **optional sub-corpus** cell (Step 2b).
> - The model + dataset are public on the Hub; the **code repo must be public** for the `git clone` in Step 1 (this notebook reuses the repo's harness — it does not reimplement it).

## Step 0 — Check the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — set Runtime -> GPU (A100/L4)'
!df -h /content | tail -1


## Step 1 — Install dependencies & clone the repo

`ripgrep` (the agent's search tool), vLLM 0.17 + the Qwen3.5 transformers build (same recipe as `TRAINING_ENV.md`), and the GrepSeek repo (we **reuse** its `inference/` harness and `rl/serve_rl.sh`).

In [ ]:
# 1a. ripgrep — the shell tool the agent calls
!apt-get -qq update && apt-get -qq install -y ripgrep >/dev/null && rg --version | head -1

# 1b. serving + client stack (this pulls a self-consistent vLLM tree)
!pip -q install vllm==0.17.0 openai huggingface_hub

# 1b2. CRITICAL: vLLM 0.17 pulls an OLD huggingface_hub, but the Qwen3.5
#       transformers-git below imports huggingface_hub.is_offline_mode (needs >=1.x).
#       Force the hub + companions to the versions the recipe uses, --no-deps,
#       BEFORE installing transformers-git. (Fixes the is_offline_mode ImportError.)
!pip -q install --no-deps 'huggingface_hub==1.8.0' 'tokenizers==0.22.2' 'safetensors==0.7.0'

# 1c. pin the Qwen3.5-capable transformers build (matches the repo's verified recipe).
#     --no-deps so it doesn't refight vLLM's pins. If vLLM 0.17 already serves
#     Qwen3.5 on your runtime you can skip this; it's here for parity with the paper env.
!pip -q install --no-deps 'transformers @ git+https://github.com/huggingface/transformers.git@f048e845684894fe60440bb8506f26ffaf7b69ac'

# 1d. the GrepSeek repo (reused harness; must be a public repo)
![ -d grepseek ] || git clone --depth 1 https://github.com/alirezasalemi7/grepseek
import sys; sys.path.insert(0, '/content/grepseek')
print('repo + deps ready')


## Step 2 — Download the full Wikipedia corpus (~14 GB)

Reuses the repo's downloader (`PeterJinGo/wiki-18-corpus` → `wiki_corpus.jsonl`). ~5 GB download, decompresses to ~14 GB. This is the part to watch on Colab; if it's too slow/large, skip to **Step 2b**.

In [ ]:
!python grepseek/sft/data_generation/download_corpus.py --dest /content/data/wiki_18_corpus
CORPUS_DIR = '/content/data/wiki_18_corpus'
!ls -lh /content/data/wiki_18_corpus/wiki_corpus.jsonl


### Step 2b (optional) — sub-corpus fallback

Only if the full corpus is too slow or disk is tight. This keeps the first N passages so `rg` is fast. **Note:** answers to arbitrary questions may not be in a small slice — this is for a quick smoke, not faithful eval. Skip if Step 2 worked.

In [ ]:
# Uncomment to use a sub-corpus instead of the full one.
# N = 2_000_000   # ~ first 2M of 21M passages
# import os; os.makedirs('/content/data/wiki_18_corpus_small', exist_ok=True)
# !head -n {N} /content/data/wiki_18_corpus/wiki_corpus.jsonl > /content/data/wiki_18_corpus_small/wiki_corpus.jsonl
# CORPUS_DIR = '/content/data/wiki_18_corpus_small'
# !ls -lh {CORPUS_DIR}/wiki_corpus.jsonl


## Step 3 — Serve the GRPO model with vLLM

Pick a serving mode with the **`SERVE_MODE`** variable below:

- **`"bf16"`** (default) — full precision, reuses `rl/serve_rl.sh` unchanged. Needs an **A100 (40 GB)** or **L4 (24 GB)**. This is what the paper numbers use.
- **`"4bit"`** — on-the-fly **bitsandbytes** 4-bit so the 9B model fits a **free T4 (16 GB)**. Same Qwen3 reasoning/tool-calling flags as `serve_rl.sh`; it just adds `--quantization bitsandbytes`. Slower (eager mode) and smaller context — good enough for a demo, not for faithful benchmark eval.

Either way the cell launches vLLM in the background and waits until it answers. First run downloads the ~18 GB model, so expect a few minutes.

In [ ]:
import os, subprocess, time, urllib.request, shlex
os.chdir('/content/grepseek')           # serve_rl.sh must run from the repo root (idempotent)
MODEL = 'alireza7/GrepSeek-Qwen3.5-9B-GRPO'
PORT  = 8000

# ── Pick ONE serving mode for your GPU — flip this single variable ───────────
#   "bf16" : full precision. Needs an A100 (40 GB) or L4 (24 GB). Fastest, and
#            this is what the paper numbers use. Reuses rl/serve_rl.sh unchanged.
#   "4bit" : on-the-fly bitsandbytes 4-bit (weights ~5-6 GB) so it fits a free
#            T4 (16 GB). Slower (forces eager mode) and smaller context — fine
#            for a demo, NOT for faithful benchmark eval.
SERVE_MODE = "bf16"
# ─────────────────────────────────────────────────────────────────────────────

logf = open('/content/vllm_server.log', 'w')

if SERVE_MODE == "bf16":
    # Reuse the repo's serve script as-is; just size it for a single GPU.
    # (If an L4 OOMs, lower MAX_MODEL_LEN or MAX_NUM_SEQS.)
    env = {**os.environ,
           'MODEL_PATH': MODEL, 'TP_SIZE': '1', 'PORT': str(PORT), 'HOST': '127.0.0.1',
           'GPU_UTIL': '0.92', 'MAX_MODEL_LEN': '16384', 'MAX_NUM_SEQS': '4',
           'SERVED_MODEL_NAME': 'grepseek'}
    server = subprocess.Popen(['bash', 'rl/serve_rl.sh'], env=env,
                              stdout=logf, stderr=subprocess.STDOUT)

elif SERVE_MODE == "4bit":
    # serve_rl.sh has no quantization knob, so build the SAME vllm command here
    # and add on-the-fly bitsandbytes 4-bit. Every other flag is identical to
    # serve_rl.sh, so the agent's behavior is the same — only weights/KV shrink.
    subprocess.run(['pip', '-q', 'install', 'bitsandbytes'], check=True)
    vllm_cmd = [
        'vllm', 'serve', MODEL,
        '--host', '127.0.0.1', '--port', str(PORT),
        '--served-model-name', 'grepseek',
        '--tensor-parallel-size', '1',
        '--max-model-len', '8192',          # smaller ctx to keep KV cache on a 16 GB T4
        '--max-num-seqs', '2',
        '--gpu-memory-utilization', '0.92',
        '--quantization', 'bitsandbytes',   # <-- the only real difference
        '--load-format', 'bitsandbytes',
        '--reasoning-parser', 'qwen3',      # everything below mirrors serve_rl.sh
        '--language-model-only',
        '--trust-remote-code',
        '--enable-auto-tool-choice',
        '--tool-call-parser', 'qwen3_coder',
    ]
    print('Command:', ' '.join(shlex.quote(c) for c in vllm_cmd))
    server = subprocess.Popen(vllm_cmd, env={**os.environ},
                              stdout=logf, stderr=subprocess.STDOUT)
else:
    raise ValueError(f"SERVE_MODE must be 'bf16' or '4bit', got {SERVE_MODE!r}")

print(f'launched vLLM ({SERVE_MODE}, pid={server.pid}); waiting for http://127.0.0.1:{PORT} ...')
ready = False
for i in range(120):  # up to ~20 min (first run downloads ~18 GB; 4-bit re-quantizes on load)
    if server.poll() is not None:
        print('server exited early — tail of log:'); print(open('/content/vllm_server.log').read()[-3000:]); break
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/v1/models', timeout=5); ready = True; break
    except Exception:
        time.sleep(10);  print(f'  ...loading ({(i+1)*10}s)', end='\r')
print('\nSERVER READY' if ready else '\nnot ready — check /content/vllm_server.log')


## Step 4 — Wire the repo's agent harness to the server

We **reuse** `inference.agent.run_agent_on_example` (the exact loop used for the paper's eval): it asks the served model, parses its `<tool_call>`, runs `rg` over `CORPUS_DIR` via the repo's `tools.run_tool`, feeds back `<tool_response>`, and repeats until `<answer>`.

In [ ]:
from openai import OpenAI
from transformers import AutoTokenizer
from inference.agent import run_agent_on_example   # reused harness

client = OpenAI(base_url=f'http://127.0.0.1:{PORT}/v1', api_key='EMPTY')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print('agent ready; corpus =', CORPUS_DIR)


## Step 5 — Ask GrepSeek a question

Edit the question and re-run. `ask()` calls the reused harness and pretty-prints the full trajectory (reasoning → shell command → retrieved snippet → answer).

### What a successful run looks like

Below is a **real recorded trajectory** of the released GRPO model (`alireza7/GrepSeek-Qwen3.5-9B-GRPO`) on the Bamboogle question *"In what country is the second tallest statue in the world?"* — a two-hop question (find the statue → find its country). The next cell asks the live server the **same** question; your run should look similar (sampling is stochastic at `temperature=0.6`).

> **Q:** In what country is the second tallest statue in the world?
> **✅ Correct answer: `China`**  · 2 tool calls, EM = 1
>
> ```text
> [assistant] <think>
>   I need to identify the second tallest statue in the world to determine its
>   country. I will start by querying the corpus for the phrase "second tallest
>   statue" to see if the relevant entry is indexed under that description.
> </think>
> <tool_call> rg -F "second tallest statue" corpus.jsonl | head -n 8
> [tool] -> "Vairocana ... The Spring Temple Buddha of Lushan County, Henan, China,
>           with a height of 126 meters, is the second tallest statue in the world ..."
>
> [assistant] <think>
>   The result identifies the Spring Temple Buddha in China, but it may be stale or
>   context-specific. To be safe I'll verify by searching "tallest statue in the
>   world" and filtering for "second".
> </think>
> <tool_call> rg -F "tallest statue in the world" corpus.jsonl | rg -i -F "second" | head -n 12
> [tool] -> "Spring Temple Buddha ... it is the second-tallest statue in the world
>           (after the Statue of Unity in Gujarat, India ...) ... located in the
>           Zhaocun township of Lushan County, Henan, China ..."
>
> [assistant] <think>
>   Both searches agree: the Spring Temple Buddha is the second-tallest statue, and
>   it is located in Lushan County, Henan, China.
> </think>
> <answer> China </answer>
> ```
>
> Notice the agent *cross-checks* its first hit with a second, differently-phrased search before answering — exactly the lexical, verifiable retrieval that Direct Corpus Interaction is designed for.

In [ ]:
import json
from inference.scoring import score   # reuse the repo's EM/F1 scorer

def ask(question, gold=None, max_turns=6, temperature=0.6):
    rec = run_agent_on_example(
        {'id': 'q', 'question': question, 'golden_answers': gold or []},
        client=client, model='grepseek', tokenizer=tokenizer,
        corpus_dir=CORPUS_DIR, max_assistant_turns=max_turns,
        temperature=temperature,
    ).to_dict()
    print('Q:', question, '\n' + '='*80)
    for m in rec['messages']:
        role = m.get('role')
        if role == 'assistant':
            print('\n[assistant]\n' + (m.get('content') or '').strip())
        elif role == 'tool':
            try:
                obj = json.loads(m.get('content') or '{}')
                out = (obj.get('stdout') or '').strip()
                print('  $ ' + (obj.get('command') or ''))
                print('  -> ' + (out[:500] + (' ...[truncated]' if len(out) > 500 else '')))
            except Exception:
                print('  [tool] ' + (m.get('content') or '')[:500])
    print('\n' + '='*80)
    print(f"ANSWER: {rec['prediction']!r}   "
          f"(turns={rec['n_assistant_turns']}, tool_calls={rec['n_tool_calls']}, "
          f"{rec['total_time_s']:.1f}s)")
    if gold:  # we know the correct answer here, so grade it with the repo's scorer
        s = score(rec['prediction'] or '', gold)
        print(f"GOLD:   {gold}   ->   "
              f"{'✅ correct (EM=1)' if s['em'] else '❌ no exact match'}, token-F1={s['f1']:.2f}")
    return rec

# Verified success for the GRPO model — same question as the recorded run above.
_ = ask('In what country is the second tallest statue in the world?', gold=['China'])


### Try your own
GrepSeek is strongest on multi-hop / exact-entity questions. A few to try:

In [ ]:
# More two-hop questions the GRPO model answers correctly (gold passed so it self-grades):
_ = ask('Who is the father of the father of observational astronomy?', gold=['Vincenzo Galilei'])
# _ = ask('When was the company that built the first steam locomotive to carry passengers '
#         'on a public rail line founded?', gold=['1823'])
# _ = ask('YOUR QUESTION HERE')   # gold optional — omit it and just read the answer


## Notes

- **Speed:** plain `rg` over 14 GB on Colab CPU is the bottleneck (~10–30 s/call). The repo also ships a **sharded-parallel engine + search daemon** (`inference/parallel_search/`) that makes this ms/query — see [`inference/README.md`](https://github.com/alirezasalemi7/grepseek/tree/main/inference). It's overkill for a single-question demo but worth it for benchmark eval.
- **Benchmark eval (EM/F1):** use the repo's `inference/run.py --datasets ...` against the same server instead of `ask()`.
- **SFT model:** swap `MODEL = 'alireza7/GrepSeek-Qwen3.5-9B-SFT'` in Step 3 to compare the pre-RL policy.
- **Shut down the server:** `server.terminate()`.

```python
# server.terminate()
```